# Prepare Live Segmentation Adaptation Dataset

Build a webcam/live-domain segmentation dataset from saved diagnostic frames. This is the seed dataset for fine-tuning the strong segmentation model on real live-camera conditions.

In [1]:
from pathlib import Path
import sys
import json

import pandas as pd
from IPython.display import display  

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'backend').exists() and (candidate / 'notebooks').exists():
            return candidate
    raise RuntimeError('Could not locate project root.')

PROJECT_ROOT = find_project_root()
BACKEND_ROOT = PROJECT_ROOT / 'backend'
if str(BACKEND_ROOT) not in sys.path:
    sys.path.append(str(BACKEND_ROOT))

from app.ml.live_segmentation import (
    LIVE_SEGMENTATION_DATASET_DIR,
    LIVE_SEGMENTATION_INPUT_DIR,
    build_live_segmentation_seed_records,
    export_live_segmentation_split,
    split_live_segmentation_seed_records,
)

PROJECT_ROOT

WindowsPath('D:/Projects/Personal Projects/Hairstyle Recommender Live Tryon')

In [2]:
INPUT_DIR = LIVE_SEGMENTATION_INPUT_DIR
OUTPUT_DIR = LIVE_SEGMENTATION_DATASET_DIR
USE_FACE_ROI = True
KEEP_ONLY_SUPPORTED_FRAMES = False
REQUIRE_RELIABLE_MASKS = True
TRAIN_RATIO = 0.7
VAL_RATIO = 0.15
SEED = 42

print('Input dir:', INPUT_DIR)
print('Output dir:', OUTPUT_DIR)

Input dir: D:\Projects\Personal Projects\Hairstyle Recommender Live Tryon\backend\outputs\live_segmentation_eval\inputs
Output dir: D:\Projects\Personal Projects\Hairstyle Recommender Live Tryon\backend\data\datasets\live_segmentation_adaptation


In [3]:
seed_payload = build_live_segmentation_seed_records(
    INPUT_DIR,
    OUTPUT_DIR,
    use_face_roi=USE_FACE_ROI,
    keep_only_supported_frames=KEEP_ONLY_SUPPORTED_FRAMES,
)

seed_records = seed_payload['records']
seed_summary = seed_payload['summary']
print(json.dumps(seed_summary, indent=2))

seed_df = pd.DataFrame(seed_records)
display(seed_df.head())

{
  "total_frames": 31,
  "supported_frames": 31,
  "reliable_masks": 2,
  "supported_ratio": 1.0,
  "reliable_ratio": 0.06451612903225806,
  "support_reason_counts": {
    "supported": 31
  },
  "mask_reason_counts": {
    "mask_too_narrow": 26,
    "reliable": 2,
    "mask_off_center": 2,
    "empty": 1
  }
}


,image_path,mask_path,source_id,source_dataset,face_detected,frame_supported,support_reason,mask_quality_reason,mask_reliable,mask_nonzero_ratio,face_width_ratio,brightness_mean,yaw_proxy,pitch_proxy,roll_degrees
0,backend/outputs/live_segmentation_eval/inputs/...,backend/data/datasets/live_segmentation_adapta...,live_frame_0001,Live-Webcam-Pseudo,True,True,supported,mask_too_narrow,False,0.007780,0.228125,58.422359,0.0646,-0.3376,1.524
1,backend/outputs/live_segmentation_eval/inputs/...,backend/data/datasets/live_segmentation_adapta...,live_frame_0002,Live-Webcam-Pseudo,True,True,supported,reliable,True,0.015023,0.245312,67.779556,0.0407,-0.3397,0.537
2,backend/outputs/live_segmentation_eval/inputs/...,backend/data/datasets/live_segmentation_adapta...,live_frame_0003,Live-Webcam-Pseudo,True,True,supported,reliable,True,0.012396,0.250000,74.241753,0.0422,-0.3391,0.076
3,backend/outputs/live_segmentation_eval/inputs/...,backend/data/datasets/live_segmentation_adapta...,live_frame_0004,Live-Webcam-Pseudo,True,True,supported,mask_too_narrow,False,0.000010,0.251563,84.863388,-0.0057,-0.3398,3.376
4,backend/outputs/live_segmentation_eval/inputs/...,backend/data/datasets/live_segmentation_adapta...,live_frame_0005,Live-Webcam-Pseudo,True,True,supported,mask_too_narrow,False,0.005521,0.268750,89.379692,0.0528,-0.3380,0.348


In [4]:
split_records = split_live_segmentation_seed_records(
    seed_records,
    train_ratio=TRAIN_RATIO,
    val_ratio=VAL_RATIO,
    seed=SEED,
    require_reliable_masks=REQUIRE_RELIABLE_MASKS,
)

export_paths = export_live_segmentation_split(split_records, OUTPUT_DIR)
print('Exported:')
for key, value in export_paths.items():
    print(f'  {key}: {value}')

summary_path = export_paths['summary']
print('\nSplit summary:')
print(summary_path.read_text(encoding='utf-8'))

Exported:
  train: D:\Projects\Personal Projects\Hairstyle Recommender Live Tryon\backend\data\datasets\live_segmentation_adaptation\train.jsonl
  val: D:\Projects\Personal Projects\Hairstyle Recommender Live Tryon\backend\data\datasets\live_segmentation_adaptation\val.jsonl
  test: D:\Projects\Personal Projects\Hairstyle Recommender Live Tryon\backend\data\datasets\live_segmentation_adaptation\test.jsonl
  summary: D:\Projects\Personal Projects\Hairstyle Recommender Live Tryon\backend\data\datasets\live_segmentation_adaptation\split_summary.json

Split summary:
{
  "train_records": 1,
  "val_records": 0,
  "test_records": 1,
  "total_records": 2
}


In [5]:
for split_name in ('train', 'val', 'test'):
    split_df = pd.DataFrame(split_records[split_name])
    print(f'\n{split_name.upper()} | rows={len(split_df)}')
    if not split_df.empty:
        display(split_df[['source_id', 'support_reason', 'mask_quality_reason', 'mask_reliable', 'mask_nonzero_ratio']].head())


TRAIN | rows=1


,source_id,support_reason,mask_quality_reason,mask_reliable,mask_nonzero_ratio
0,live_frame_0003,supported,reliable,True,0.012396



VAL | rows=0

TEST | rows=1


,source_id,support_reason,mask_quality_reason,mask_reliable,mask_nonzero_ratio
0,live_frame_0002,supported,reliable,True,0.015023
